# 00 — Data test: adaptive LTV for tokenized real estate

Purpose: verify every input the model needs **before** any modelling. Each test fetches the data,
checks date range, frequency and gaps, prints PASS / PARTIAL / FAIL, and saves a raw snapshot.
After this notebook, the project reads only the frozen files in `data/raw/` (Case-Shiller is the
exception: S&P restricts redistribution, so it is cached in `data/cache/`, which is git-ignored).

Rules decided before running:
- Every collection is timed. If a paginated collection projects to more than 30 minutes, it stops
  and the input goes to its declared fallback.
- An input that fails by 12 Nov 2026 uses its fallback. No revisiting afterwards.

Run order: top to bottom. Each test is independent; a failure in one does not stop the others.

## Setup

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from web3 import Web3

CUTOFF = "2026-11-10"           # fixed data cutoff for reproducibility
MAX_COLLECTION_MIN = 30         # time budget per paginated collection

RAW = Path("data/raw")
CACHE = Path("data/cache")      # git-ignored (licensed data)
RAW.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

GNOSIS_RPC = "https://rpc.gnosischain.com"
BLOCKSCOUT = "https://gnosis.blockscout.com/api"

# armmWXDAI, the RMM v3 xDAI deposit token (source: Aave governance TEMP CHECK, 18 Jun 2025).
# Pool and data-provider addresses are derived on-chain from it instead of being hard-coded.
ARMM_WXDAI = Web3.to_checksum_address("0x0cA4f5554Dd9Da6217d62D8df2816c82bba4157b")

session = requests.Session()
session.headers["User-Agent"] = "adaptive-ltv-data-test"

RESULTS = {}

def record(name, status, note):
    RESULTS[name] = {"status": status, "note": note}
    print(f"[{status}] {name}: {note}")

def get_json(url, params=None, timeout=60):
    r = session.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

## 1. Detroit house prices (FRED `DEXRNSA`)

Needed for: volatility calibration and the market-shock scenario.
Pass criteria: monthly, starts by 1991-01, ends at or after 2026-06, no missing months.

In [ ]:
try:
    t0 = time.time()
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXRNSA"
    cs = pd.read_csv(url)
    cs.columns = ["date", "dexrnsa"]
    cs["date"] = pd.to_datetime(cs["date"])
    cs["dexrnsa"] = pd.to_numeric(cs["dexrnsa"], errors="coerce")
    cs = cs.dropna().set_index("date")
    cs.to_csv(CACHE / "dexrnsa.csv")

    expected = pd.date_range(cs.index.min(), cs.index.max(), freq="MS")
    missing = expected.difference(cs.index)
    peak_date = cs["dexrnsa"].idxmax()
    print(f"range {cs.index.min():%Y-%m} to {cs.index.max():%Y-%m}, {len(cs)} obs, "
          f"{len(missing)} missing months, {time.time()-t0:.1f}s")
    print(f"peak {cs['dexrnsa'].max():.2f} in {peak_date:%Y-%m}")
    print(cs.loc["2005-12":"2011-04"].iloc[[0, -1]])

    ok = (cs.index.min() <= pd.Timestamp("1991-01-01")
          and cs.index.max() >= pd.Timestamp("2026-06-01") and len(missing) == 0)
    record("case_shiller", "PASS" if ok else "PARTIAL",
           f"{cs.index.min():%Y-%m}..{cs.index.max():%Y-%m}, missing={len(missing)}")
except Exception as e:
    record("case_shiller", "FAIL", repr(e))

## 2. Pool liquidity and utilization (DefiLlama)

Needed for: the liquidity input and calibration of the liquidity crunch.
For lending protocols DefiLlama reports TVL as supplied minus borrowed, so the TVL held in
stablecoins is the idle stablecoin liquidity. Stablecoin utilization is then
`borrowed_stable / (borrowed_stable + tvl_stable)`.

Pass criteria: daily series covering at least 2024-02 (RMM v3 launch) to 2026-07, with token
breakdown for the stablecoins. Fallback: on-chain `ReserveDataUpdated` events (not run here).

In [ ]:
STABLES = {"USDC", "WXDAI", "XDAI", "USDC.E"}

def token_frame(series):
    """DefiLlama tokensInUsd -> DataFrame (date x token)."""
    rows = [{"date": pd.to_datetime(p["date"], unit="s"), **p["tokens"]} for p in series]
    return pd.DataFrame(rows).set_index("date").sort_index()

try:
    t0 = time.time()
    llama = get_json("https://api.llama.fi/protocol/realt-rmm-marketplace-v2")
    (RAW / "defillama_rmm.json").write_text(json.dumps(llama))
    print("chainTvls keys:", list(llama["chainTvls"].keys()), f"({time.time()-t0:.1f}s)")

    tvl = token_frame(llama["chainTvls"]["xDai"]["tokensInUsd"])
    bor = token_frame(llama["chainTvls"]["xDai-borrowed"]["tokensInUsd"])
    print("TVL tokens:", list(tvl.columns))
    print("borrowed tokens:", list(bor.columns))

    pick = lambda df: df[[c for c in df.columns if c.upper() in STABLES]].sum(axis=1)
    liq = pd.DataFrame({"tvl_stable": pick(tvl), "borrowed_stable": pick(bor)}).dropna()
    liq["utilization"] = liq["borrowed_stable"] / (liq["borrowed_stable"] + liq["tvl_stable"])
    liq = liq.loc[:CUTOFF]
    liq.to_csv(RAW / "rmm_liquidity_daily.csv")

    print(liq.resample("MS").last().loc["2025-06":].round(3))
    ok = liq.index.min() <= pd.Timestamp("2024-03-01") and liq.index.max() >= pd.Timestamp("2026-07-01")
    record("pool_liquidity", "PASS" if ok else "PARTIAL",
           f"{liq.index.min():%Y-%m-%d}..{liq.index.max():%Y-%m-%d}, {len(liq)} days")
except Exception as e:
    record("pool_liquidity", "FAIL", repr(e))

## 3. Risk parameters (RMM v3, on-chain)

Needed for: the fixed-parameter benchmark (LTV, liquidation threshold, liquidation bonus).
Path: armmWXDAI -> `POOL()` -> `ADDRESSES_PROVIDER()` -> `getPoolDataProvider()`, then
`getReserveConfigurationData` for every reserve, read at the latest block and compared with
the LlamaRisk values for June 2025 (past state would need an archive node).

Pass criteria: collateral reserve(s) with LTV and liquidation threshold readable.
Fallback: LlamaRisk values (60% LTV, 70% LT, 10% penalty, June 2025), cited.

In [ ]:
w3 = Web3(Web3.HTTPProvider(GNOSIS_RPC, request_kwargs={"timeout": 60}))

ABI_ATOKEN = [{"name": "POOL", "inputs": [], "outputs": [{"type": "address"}],
               "stateMutability": "view", "type": "function"}]
ABI_POOL = [
    {"name": "ADDRESSES_PROVIDER", "inputs": [], "outputs": [{"type": "address"}],
     "stateMutability": "view", "type": "function"},
    {"name": "getReservesList", "inputs": [], "outputs": [{"type": "address[]"}],
     "stateMutability": "view", "type": "function"},
]
ABI_PROVIDER = [{"name": "getPoolDataProvider", "inputs": [], "outputs": [{"type": "address"}],
                 "stateMutability": "view", "type": "function"}]
ABI_DP = [{"name": "getReserveConfigurationData",
           "inputs": [{"name": "asset", "type": "address"}],
           "outputs": [{"name": n, "type": t} for n, t in [
               ("decimals", "uint256"), ("ltv", "uint256"), ("liquidationThreshold", "uint256"),
               ("liquidationBonus", "uint256"), ("reserveFactor", "uint256"),
               ("usageAsCollateralEnabled", "bool"), ("borrowingEnabled", "bool"),
               ("stableBorrowRateEnabled", "bool"), ("isActive", "bool"), ("isFrozen", "bool")]],
           "stateMutability": "view", "type": "function"}]
ABI_ERC20 = [{"name": "symbol", "inputs": [], "outputs": [{"type": "string"}],
              "stateMutability": "view", "type": "function"}]

def read_params(block="latest"):
    rows = []
    for asset in pool.functions.getReservesList().call(block_identifier=block):
        cfg = dp.functions.getReserveConfigurationData(asset).call(block_identifier=block)
        try:
            sym = w3.eth.contract(asset, abi=ABI_ERC20).functions.symbol().call()
        except Exception:
            sym = "?"
        rows.append({"asset": asset, "symbol": sym, "ltv": cfg[1] / 1e4,
                     "liq_threshold": cfg[2] / 1e4, "liq_bonus": cfg[3] / 1e4 - 1,
                     "collateral": cfg[5], "borrowable": cfg[6], "frozen": cfg[9]})
    return pd.DataFrame(rows)

try:
    t0 = time.time()
    POOL = w3.eth.contract(ARMM_WXDAI, abi=ABI_ATOKEN).functions.POOL().call()
    pool = w3.eth.contract(POOL, abi=ABI_POOL)
    provider = pool.functions.ADDRESSES_PROVIDER().call()
    DATA_PROVIDER = w3.eth.contract(provider, abi=ABI_PROVIDER).functions.getPoolDataProvider().call()
    dp = w3.eth.contract(DATA_PROVIDER, abi=ABI_DP)
    print(f"Pool {POOL}\nDataProvider {DATA_PROVIDER}")

    params_now = read_params()
    params_now.to_csv(RAW / "rmm_params_latest.csv", index=False)
    print(params_now.to_string(index=False), f"\n({time.time()-t0:.1f}s)")
    record("risk_params", "PASS" if (params_now["ltv"] > 0).any() else "PARTIAL",
           "latest block read; check collateral rows above")
except Exception as e:
    record("risk_params", "FAIL", repr(e))

# June 2025 cross-check: reading past state needs an archive node, so instead the
# latest on-chain values are compared with LlamaRisk (60% LTV, 70% LT, 10% bonus, June 2025).
rtw = params_now[params_now["symbol"].str.startswith("RTW")]
match = ((rtw["ltv"].round(2) == 0.60) & (rtw["liq_threshold"].round(2) == 0.70)
         & (rtw["liq_bonus"].round(2) == 0.10)).all()
record("risk_params_vs_llamarisk", "PASS" if match else "PARTIAL",
       "on-chain RTW params match LlamaRisk June 2025" if match else "differ from LlamaRisk, check")

## 4. Liquidations (RMM v3 `LiquidationCall` events, Gnosis public RPC)

Needed for: calibrating liquidation behaviour and checking the stress period.
Etherscan dropped Gnosis from its free tier and the Blockscout legacy endpoint now redirects to
Gnosisscan (403 on the first run), so the events are read with `eth_getLogs` on the public RPC.
The block range adapts: it halves when the RPC rejects a range and grows after successes.
Collection starts at 2024-02-01 (RMM v3 launch) and stops if the projection exceeds the budget.

Pass criteria: events readable through at least 2026-01. Fallback: LlamaRisk count (39 events,
US$ 12,849 until June 2025), cited, with no series after that.

In [ ]:
TOPIC_LIQ = Web3.keccak(text="LiquidationCall(address,address,address,uint256,uint256,address,bool)")

def block_at(ts):
    """First block with timestamp >= ts (binary search on block headers, no archive needed)."""
    lo, hi = 1, w3.eth.block_number
    while lo < hi:
        mid = (lo + hi) // 2
        if w3.eth.get_block(mid)["timestamp"] < ts:
            lo = mid + 1
        else:
            hi = mid
    return lo

try:
    t0 = time.time()
    start = block_at(pd.Timestamp("2024-02-01").timestamp())
    end = block_at(pd.Timestamp(CUTOFF).timestamp())
    print(f"blocks {start}..{end}")

    logs, cur, step = [], start, 500_000
    while cur <= end:
        to = min(cur + step - 1, end)
        try:
            logs += w3.eth.get_logs({"address": POOL, "topics": [TOPIC_LIQ],
                                     "fromBlock": cur, "toBlock": to})
            cur, step = to + 1, min(step * 2, 2_000_000)
        except Exception as err:
            if step <= 1_000:
                raise
            step //= 2
            continue
        done = (cur - start) / (end - start + 1)
        projected = (time.time() - t0) / max(done, 1e-9) / 60
        if projected > MAX_COLLECTION_MIN:
            raise TimeoutError(f"projected {projected:.0f} min > budget")

    blk_ts = {}
    def ts_of(n):
        if n not in blk_ts:
            blk_ts[n] = w3.eth.get_block(n)["timestamp"]
        return blk_ts[n]

    liq_ev = pd.DataFrame([{
        "block": l["blockNumber"],
        "date": pd.to_datetime(ts_of(l["blockNumber"]), unit="s"),
        "tx": l["transactionHash"].hex(),
        "collateral_asset": "0x" + l["topics"][1].hex()[-40:],
        "debt_asset": "0x" + l["topics"][2].hex()[-40:],
        "user": "0x" + l["topics"][3].hex()[-40:],
        "data": l["data"].hex(),
    } for l in logs])
    if len(liq_ev):
        liq_ev = liq_ev.sort_values("block")
    liq_ev.to_csv(RAW / "rmm_liquidations.csv", index=False)

    print(f"{len(liq_ev)} events in {time.time()-t0:.0f}s")
    if len(liq_ev):
        print(liq_ev.set_index("date").resample("QS").size().to_string())
    ok = len(liq_ev) > 0 and liq_ev["date"].max() >= pd.Timestamp("2026-01-01")
    record("liquidations", "PASS" if ok else "PARTIAL",
           f"{len(liq_ev)} events, last {liq_ev['date'].max() if len(liq_ev) else None}")
except Exception as e:
    record("liquidations", "FAIL", repr(e)[:300])

## 5. Collateral price, appraisal history and rent (RealT community API)

First run (10 Nov 2026): `api.realt.community` does not resolve (DNS failure). Kept here so the
failure is documented; if it stays down, appraisal and rent use the declared fallback.

Needed for: the oracle price (appraisal steps) and the rent-coverage input.
`/v1/token` gives the current state of every property; `/v1/tokenHistory` (if public) gives the
history of price and rent changes per token. If the history exists, it covers both appraisal
steps and rent changes and replaces the rent-tracker files.

Pass criteria: history with dated price and rent changes through 2026. Fallback: stylized
appraisal steps every 1 to 2 years and the documented rent sequence, both declared as stylized.

In [ ]:
REALT_API = "https://api.realt.community/v1"

try:
    t0 = time.time()
    tokens = get_json(f"{REALT_API}/token")
    (RAW / "realt_tokens.json").write_text(json.dumps(tokens))
    tok = pd.DataFrame(tokens)
    print(f"{len(tok)} tokens ({time.time()-t0:.1f}s)")
    print("fields:", list(tok.columns))
    cols = [c for c in ["fullName", "tokenPrice", "netRentMonth", "rentedUnits", "totalUnits",
                        "rentStartDate", "initialLaunchDate"] if c in tok.columns]
    print(tok[cols].head().to_string())
    record("realt_tokens", "PASS", f"{len(tok)} tokens, current snapshot only")
except Exception as e:
    record("realt_tokens", "FAIL", repr(e))

try:
    t0 = time.time()
    hist = get_json(f"{REALT_API}/tokenHistory")
    (RAW / "realt_token_history.json").write_text(json.dumps(hist))
    print(f"history entries: {len(hist)} ({time.time()-t0:.1f}s)")
    print(json.dumps(hist[0], indent=1)[:1500])
    record("realt_history", "PARTIAL", "downloaded; inspect structure above before deciding")
except Exception as e:
    record("realt_history", "FAIL", repr(e))

## 6. Rent tracker raw files (manual check)

The RealToken rent tracker states it uses the weekly RealT master rent files from 2021-02-28 to
2026-02-01. This test only checks that the site responds. Obtaining the raw files is a manual
step: look for a download link or ask the author. Only needed if section 5 has no rent history.

In [ ]:
try:
    r = session.get("https://ehpst.duckdns.org/realt_rent_tracker/help", timeout=30)
    record("rent_tracker", "PARTIAL", f"site HTTP {r.status_code}; raw files need a manual request")
except Exception as e:
    record("rent_tracker", "FAIL", repr(e))

## Summary

In [ ]:
summary = pd.DataFrame(RESULTS).T
summary.to_csv(RAW / "data_test_summary.csv")
summary